# RAPIDS Visualization Guide Base Notebook

This fixture provides the dataset setup, cleaning steps, and visualization goal for an updated RAPIDS visualization guide. It intentionally stops before the completed visualization/dashboard implementation so an eval can test construction from context.

Goal: build a maintained GPU-accelerated visualization guide for Divvy bike share data using cuDF or a backend-aware CPU/GPU path, hvPlot/HoloViews/Datashader/Panel for notebook-first exploration, and Dash only if a standalone app is requested. Do not use cuxfilter.

## Dependencies

Expected maintained stack for generated work:

- cudf, optionally with pandas fallback
- hvplot, holoviews, panel, datashader, colorcet
- cugraph/cuml only for graph analytics or clustering sections
- plotly/dash only for standalone app examples

The construction eval should preserve the data preparation intent while replacing historical cuxfilter dashboard patterns with maintained libraries.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import os
import urllib.request

try:
    import cudf
except Exception:
    cudf = None

import pandas as pd

## Download Dataset

The Divvy Bike Share dataset is public. Keep downloads disabled by default for evals; agents can explain how to run the cell or adapt it to a local fixture.

In [ ]:
S3 = "https://divvy-tripdata.s3.amazonaws.com/"
DATA_DIR = Path("data")
YEARS = range(2021, 2022)
MONTHS = range(1, 13)
DOWNLOAD_DATA = False

DATA_DIR.mkdir(parents=True, exist_ok=True)

if DOWNLOAD_DATA:
    for year in YEARS:
        for month in MONTHS:
            file_name = f"{year}{month:02d}-divvy-tripdata.zip"
            url = f"{S3}{file_name}"
            zip_path = DATA_DIR / file_name
            if not zip_path.exists():
                print(f"Downloading {file_name}...")
                urllib.request.urlretrieve(url, zip_path)
            with ZipFile(zip_path) as zip_file:
                zip_file.extractall(DATA_DIR)

## Backend-Aware Loading

Use cuDF for large GPU-suitable data, but keep the backend explicit so the same notebook can be reviewed on CPU-only machines.

In [ ]:
BACKEND = "auto"  # "auto", "cudf", or "pandas"

def read_csv(path, *, backend=BACKEND):
    if backend not in {"auto", "cudf", "pandas"}:
        raise ValueError("backend must be auto, cudf, or pandas")

    if backend in {"auto", "cudf"} and cudf is not None:
        try:
            return cudf.read_csv(path), "cudf"
        except Exception:
            if backend == "cudf":
                raise

    return pd.read_csv(path), "pandas"

def concat_frames(frames, backend):
    if backend == "cudf":
        return cudf.concat(frames)
    return pd.concat(frames, ignore_index=True)

def load_trip_data(data_dir=DATA_DIR, backend=BACKEND):
    frames = []
    selected_backend = None
    for file_path in sorted(Path(data_dir).rglob("20*.csv")):
        frame, selected_backend = read_csv(file_path, backend=backend)
        frames.append(frame)
    if not frames:
        raise FileNotFoundError("No Divvy CSV files found. Download data or provide local files.")
    return concat_frames(frames, selected_backend), selected_backend

# df, backend = load_trip_data()

## Reformat and Clean Data

These cells express the intended preparation steps. Generated solutions should keep transformations explicit and mention GPU/CPU boundaries.

In [ ]:
def clean_trip_data(df, backend):
    df = df.dropna(subset=["end_lat"])
    df["start_station_name"] = df["start_station_name"].fillna("none")
    df["end_station_name"] = df["end_station_name"].fillna("none")

    min_lat, max_lat = 41.5, 42.5
    min_lng, max_lng = -88.0, -87.0
    df = df[
        (df["start_lat"] >= min_lat)
        & (df["start_lat"] <= max_lat)
        & (df["start_lng"] >= min_lng)
        & (df["start_lng"] <= max_lng)
        & (df["end_lat"] >= min_lat)
        & (df["end_lat"] <= max_lat)
        & (df["end_lng"] >= min_lng)
        & (df["end_lng"] <= max_lng)
    ]

    if backend == "cudf":
        df["started_at"] = cudf.to_datetime(df["started_at"])
        df["ended_at"] = cudf.to_datetime(df["ended_at"])
    else:
        df["started_at"] = pd.to_datetime(df["started_at"])
        df["ended_at"] = pd.to_datetime(df["ended_at"])

    df["year"] = df["started_at"].dt.year
    df["month"] = df["started_at"].dt.month
    df["day"] = df["started_at"].dt.day
    df["hour"] = df["started_at"].dt.hour
    df["day_of_week"] = df["started_at"].dt.dayofweek

    df["dur_min"] = df["ended_at"] - df["started_at"]
    df["dur_min"] = (df["dur_min"].dt.seconds / 60).round().astype("float32")

    return df.drop(
        ["ride_id", "started_at", "ended_at", "start_station_id", "end_station_id"],
        axis=1,
    ).reset_index(drop=True)

# df = clean_trip_data(df, backend)

## Construction Task

From this base, construct an updated visualization guide notebook. Expected sections include simple hvPlot charts, dense Datashader views, a notebook-first linked Panel dashboard, bounded tables, clear backend state, and optional Dash guidance only when a standalone app is appropriate. Do not import or depend on cuxfilter.